<a href="https://colab.research.google.com/github/chaseshallow/Ansci4040FinalClassProject/blob/main/PipelineSteps.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Calving-pen pipeline: all steps in one notebook

One notebook for the whole pipeline, **one cell per step**, each calling that step's function (the same code and output folders as the separate step notebooks). Use it for any video:

1. **Setup** and **settings** (cells 1–2): set the video once; `Test = True` runs the 4-minute "a" versions, `False` the full clip.
2. Run the step cells **in order**, checking each result before the next. To redo a step, change cell 2 and rerun that cell and the ones after it. Every step **overwrites** its folder for this video (step 03 only continues a run that stopped part-way).

| Cell | Step | Function | Model | T4 time (full hour) |
|---|---|---|---|---|
| 3 | 01 cut the clip | `PrepareClips` | ffmpeg | a few min (skip if `ReferenceClip.mp4` exists) |
| 4 | 02 detect | `DetectCows` | **YOLO26n** | ~10–15 min |
| 5 | 03 track | `TrackCows` | **SAM 2.1 Small**, video mode | ~75–90 min, resumable |
| 6 | 04 crop + mask | `CropCows` | **SAM 2.1 Small**, image mode | ~10 min |
| 7 | 05 fingerprint | `EmbedCrops` | **DINOv3 ViT-S/16** (fallback DINOv2) | ~2 min |
| 8 | 06 pool | `BuildPool` | none | seconds |

For many videos without checking each step, use `RunPipeline.ipynb`. Needs a **T4 GPU** and the `HF_TOKEN` secret (DINOv3). The folder is shared: one person runs a step at a time. Edit cell 2 and rerun; don't add cells.

In [ ]:
# 1. Setup: mount Drive, get the pipeline code from GitHub, install the model libraries.
from google.colab import drive, userdata
drive.mount("/content/drive")

import os, sys
RepoUrl = "https://github.com/chaseshallow/Ansci4040FinalClassProject.git"
RepoBranch = "main"
RepoDir = "/content/Ansci4040FinalClassProject"

if os.path.isdir(RepoDir):
    !git -C "{RepoDir}" fetch -q origin "{RepoBranch}" && git -C "{RepoDir}" checkout -q "{RepoBranch}" && git -C "{RepoDir}" pull -q --ff-only
else:
    !git clone -q --branch "{RepoBranch}" "{RepoUrl}" "{RepoDir}"
%pip install -q -r "{RepoDir}/CalvingPen/requirements.txt"
sys.path.insert(0, f"{RepoDir}/CalvingPen")

# Only needed after an update: Runtime > Restart session, then rerun from here.
from CalvingPipeline import PrepareClips
from CalvingPipeline.Detect import DetectCows
from CalvingPipeline.Track import TrackCows
from CalvingPipeline.Crop import CropCows
from CalvingPipeline.Embed import EmbedCrops
from CalvingPipeline.Pool import BuildPool
from CalvingPipeline.Pipeline import StepFolders
from CalvingPipeline.Paths import OutputRoot, RawVideoDir
import pandas as pd
from IPython.display import Image, Video, display
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4 GPU")

try:
    HfToken = userdata.get("HF_TOKEN")   # from Colab Secrets; never paste the token here
except Exception:
    HfToken = None
print("HF_TOKEN:", "found" if HfToken else "not set - DINOv2 fallback will be used")

In [ ]:
# 2. Settings: the only cell to edit.
VideoStem = "2026-09-26 00-00-00 Calving Pen 2"   # output folder name for this video
VideoPath = RawVideoDir / "2026-09-26 00-00-00 Calving Pen 2.avi"   # only for step 01 (videos were renamed on Drive: set the new name)
ClipStart, ClipEnd = "03:35:11", None             # only for step 01: the window with the calving (None = to the end)

Test = False           # True = 4 minutes from TestStart ("a" folders); False = the whole clip ("b" folders)
TestStart = 1080       # seconds into the clip

# Step settings (the defaults of the step notebooks)
TargetFps = 2          # 02: frames per second (the camera updates ~2 times a second)
ImageSize = 1024       # 02: YOLO picture size (640 misses the lying cow far more often)
Confidence = 0.25      # 02: lowest YOLO confidence kept
SeedConfidence = 0.4   # 03: YOLO box needed to start SAM in a chunk
ChunkSeconds = 120     # 03: SAM restarts from a YOLO box every 2 min
Preview = True         # 02 (test only) and 03: preview videos
MinFrames, MinConfidence = 10, 0.4   # 04: drop tracks shorter than 5 s or less sure
CropsPerTrack = 8 if Test else 24    # 04: at least this many crops per track
CropEverySeconds = 15  # 04: and at least one every 15 s
GroupSeconds = 60 if Test else 300   # 05: heatmap piece length
ExcludeSeconds = 60 if Test else 300 # 06: self-check may match its own track this far away

VideoDir = OutputRoot / VideoStem
ClipPath = VideoDir / "ReferenceClip.mp4"
Folder = {Step: (OutputRoot if Step == 6 else VideoDir) / Name for Step, Name in StepFolders(Test).items()}
Start, End = (TestStart, TestStart + 240) if Test else (0, None)
for Step, Path_ in Folder.items():
    print(f"step 0{Step}: {Path_}")

In [ ]:
# 3. Step 01 · Cut the clip (ffmpeg). Skip if ReferenceClip.mp4 already exists.
if ClipPath.exists():
    print("ReferenceClip.mp4 already there; delete it to cut it again")
else:
    assert VideoPath.exists(), VideoPath
    display(PrepareClips(VideoPath, VideoDir, ClipStart, ClipEnd))

In [ ]:
# 4. Step 02 · Detect cows (YOLO26n): boxes in every sampled frame.
Stats = DetectCows(ClipPath, Folder[2], Start, End, TargetFps=TargetFps, Confidence=Confidence,
                   ImageSize=ImageSize, Preview=Preview and Test)
display(pd.read_csv(Folder[2] / "DetectionSummary.csv"))
Path = Folder[2] / "PreviewDetections.mp4"
if Path.exists() and Path.stat().st_size < 50 * 2**20:   # bigger files: open them from Drive
    display(Video(str(Path), embed=True, width=960))

In [ ]:
# 5. Step 03 · Track the cow (SAM 2.1, video mode). Rerun to continue a stopped run.
Stats = TrackCows(ClipPath, Folder[2], Folder[3], ChunkSeconds=ChunkSeconds, SeedConfidence=SeedConfidence,
                  CompareTracksPath=VideoDir / "BaselineByteTrack" / "Tracks.csv", Preview=Preview)
display(pd.read_csv(Folder[3] / "Comparison.csv"))
display(pd.read_csv(Folder[3] / "TrackSummary.csv"))
Path = Folder[3] / "PreviewTracks.mp4"
if Path.exists() and Path.stat().st_size < 50 * 2**20:   # bigger files: open them from Drive
    display(Video(str(Path), embed=True, width=960))

In [ ]:
# 6. Step 04 · Cut out the cow (SAM 2.1, image mode): masked crops.
Stats = CropCows(ClipPath, Folder[3] / "Tracks.csv", Folder[4], MinFrames=MinFrames, MinConfidence=MinConfidence,
                 CropsPerTrack=CropsPerTrack, CropEverySeconds=CropEverySeconds)
display(pd.read_csv(Folder[4] / "KeptTracks.csv"))
display(Image(str(Folder[4] / "Sheets" / "Overview.jpg")))

In [ ]:
# 7. Step 05 · Fingerprint each crop (DINOv3).
Stats = EmbedCrops(Folder[4], Folder[5], VideoStem, Token=HfToken, GroupSeconds=GroupSeconds)
display(Image(str(Folder[5] / "TrackSimilarity.png")))

In [ ]:
# 8. Step 06 · Pool all videos' fingerprints and self-check (no model).
Stats = BuildPool(OutputRoot, Folder[6], EmbeddingFolder=Folder[5].name, ExcludeSeconds=ExcludeSeconds)
display(pd.read_csv(Folder[6] / "PoolSummary.csv"))
if (Folder[6] / "SelfCheck.png").exists():
    display(Image(str(Folder[6] / "SelfCheck.png")))

## About this notebook

Each cell calls one function from `CalvingPen/CalvingPipeline/`: `PrepareClips` (VideoTools.py), `DetectCows` (Detect.py), `TrackCows` (Track.py), `CropCows` (Crop.py), `EmbedCrops` (Embed.py), `BuildPool` (Pool.py). What each model does, what we found, and the glossary: `CalvingPen/README.md`.

**Check before moving on:** step 02 `CowShare` per minute (low = YOLO cannot see her); step 03 few long tracks and low `YoloDisagrees`, masks on the cow in the preview; step 04 `Overview.jpg` shows only the cow (no calf, no floor); step 05 bright blocks in the heatmap. Step 06 labels come from each video's `Identities.csv` (edit by hand; never overwritten).